In [10]:
import pandas as pd

df = pd.read_csv('../data/cleaned/ecommerce_orders_features.csv')
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(df.columns.tolist())

Loaded: 110,189 rows × 36 columns
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'total_payment_value', 'payment_installments', 'dominant_payment_type', 'review_score', 'review_answer_timestamp', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_category_name_english', 'delivery_days', 'estimated_delay_days', 'is_late_delivery', 'order_month', 'order_year', 'order_weekday', 'order_hour', 'revenue', 'freight_ratio', 'review_group', 'delivery_bucket']


In [11]:
tableau_cols = {
    'order_id':                         'order_id',
    'order_purchase_timestamp':         'order_date',
    'order_month':                      'order_month',
    'order_weekday':                    'order_weekday',
    'order_hour':                       'order_hour',
    'customer_state':                   'customer_state',
    'customer_city':                    'customer_city',
    'product_category_name_english':    'product_category',
    'price':                            'item_price',
    'freight_value':                    'freight_value',
    'revenue':                          'revenue',
    'freight_ratio':                    'freight_ratio',
    'dominant_payment_type':            'payment_type',        
    'payment_installments':             'payment_installments',
    'total_payment_value':              'payment_value',       
    'delivery_days':                    'delivery_days',
    'estimated_delay_days':             'estimated_delay_days',
    'is_late_delivery':                 'is_late_delivery',
    'delivery_bucket':                  'delivery_bucket',
    'review_score':                     'review_score',
    'review_group':                     'review_group',
}

# Keep only columns that exist in df
existing_cols = {k: v for k, v in tableau_cols.items() if k in df.columns}
tableau_df = df[list(existing_cols.keys())].rename(columns=existing_cols)

print(f"Tableau dataset: {tableau_df.shape[0]:,} rows × {tableau_df.shape[1]} columns")
print(tableau_df.columns.tolist())

Tableau dataset: 110,189 rows × 21 columns
['order_id', 'order_date', 'order_month', 'order_weekday', 'order_hour', 'customer_state', 'customer_city', 'product_category', 'item_price', 'freight_value', 'revenue', 'freight_ratio', 'payment_type', 'payment_installments', 'payment_value', 'delivery_days', 'estimated_delay_days', 'is_late_delivery', 'delivery_bucket', 'review_score', 'review_group']


In [12]:
# Ensure order_date is clean datetime string for Tableau
tableau_df['order_date'] = pd.to_datetime(tableau_df['order_date'], errors='coerce')

# Check how many nulls
print(f"Null order_date rows: {tableau_df['order_date'].isna().sum()}")

# Format as YYYY-MM-DD for Tableau compatibility
tableau_df['order_date'] = tableau_df['order_date'].dt.strftime('%Y-%m-%d')

print(tableau_df['order_date'].head())

Null order_date rows: 0
0    2017-10-02
1    2018-07-24
2    2018-08-08
3    2017-11-18
4    2018-02-13
Name: order_date, dtype: object


In [6]:
print("Unique product categories:", tableau_df['product_category'].nunique())
print("\nTop 10 categories:")
print(tableau_df['product_category'].value_counts().head(10))

# Check for nulls
print(f"\nNull product_category rows: {tableau_df['product_category'].isna().sum()}")

Unique product categories: 72

Top 10 categories:
product_category
bed_bath_table           10953
health_beauty             9465
sports_leisure            8430
furniture_decor           8160
computers_accessories     7643
housewares                6795
watches_gifts             5857
telephony                 4430
garden_tools              4268
auto                      4139
Name: count, dtype: int64

Null product_category rows: 0


In [14]:
cols_to_check = ['delivery_days', 'revenue', 'review_score', 'is_late_delivery',
                 'payment_type', 'customer_state', 'freight_ratio']

existing = [c for c in cols_to_check if c in tableau_df.columns]
missing  = [c for c in cols_to_check if c not in tableau_df.columns]

print(f"✅ Found:   {existing}")
print(f"❌ Missing: {missing}")

print("\nNull counts:")
print(tableau_df[existing].isna().sum())

print("\nValue ranges:")
range_cols = [c for c in ['delivery_days', 'revenue', 'freight_ratio', 'review_score']
              if c in tableau_df.columns]
print(tableau_df[range_cols].describe().round(2))

✅ Found:   ['delivery_days', 'revenue', 'review_score', 'is_late_delivery', 'payment_type', 'customer_state', 'freight_ratio']
❌ Missing: []

Null counts:
delivery_days         0
revenue               0
review_score        827
is_late_delivery      0
payment_type          3
customer_state        0
freight_ratio         0
dtype: int64

Value ranges:
       delivery_days    revenue  freight_ratio  review_score
count      110189.00  110189.00      110189.00     109362.00
mean           12.01     139.93           0.32          4.08
std             9.45     189.32           0.35          1.35
min             0.00       6.08           0.00          1.00
25%             6.00      55.18           0.13          4.00
50%            10.00      92.12           0.23          5.00
75%            15.00     157.47           0.39          5.00
max           209.00    6929.31          26.24          5.00


In [15]:
print(f"Final Tableau dataset shape: {tableau_df.shape}")
print(f"\nColumn dtypes:\n{tableau_df.dtypes}")
print(f"\nSample rows:")
tableau_df.head(3)

Final Tableau dataset shape: (110189, 21)

Column dtypes:
order_id                 object
order_date               object
order_month              object
order_weekday            object
order_hour                int64
customer_state           object
customer_city            object
product_category         object
item_price              float64
freight_value           float64
revenue                 float64
freight_ratio           float64
payment_type             object
payment_installments    float64
payment_value           float64
delivery_days             int64
estimated_delay_days      int64
is_late_delivery         object
delivery_bucket          object
review_score            float64
review_group             object
dtype: object

Sample rows:


,order_id,order_date,order_month,order_weekday,order_hour,customer_state,customer_city,product_category,item_price,freight_value,...,freight_ratio,payment_type,payment_installments,payment_value,delivery_days,estimated_delay_days,is_late_delivery,delivery_bucket,review_score,review_group
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02,2017-10,Monday,10,SP,sao paulo,housewares,29.99,8.72,...,0.2908,voucher,1.0,38.71,8,-8,No,Normal,4.0,High
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24,2018-07,Tuesday,20,BA,barreiras,perfumery,118.70,22.76,...,0.1917,boleto,1.0,141.46,13,-6,No,Normal,4.0,High
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08,2018-08,Wednesday,8,GO,vianopolis,auto,159.90,19.22,...,0.1202,credit_card,3.0,179.12,9,-18,No,Normal,5.0,High


In [17]:
tableau_df.to_csv('../data/cleaned/tableau_ecommerce_dashboard.csv', index=False)
print("Saved: data/cleaned/tableau_ecommerce_dashboard.csv")
print(f"Rows: {tableau_df.shape[0]:,} | Columns: {tableau_df.shape[1]}")

Saved: data/cleaned/tableau_ecommerce_dashboard.csv
Rows: 110,189 | Columns: 21
